## Notebook to demo HotSpot Analysis.

Data was download from [here](https://www.arcgis.com/home/item.html?id=71a65d35688a4502b123cbdfc99afdee). Note that the spatial reference of the data is in "ESRI:102726".

References:

- https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-statistics/what-is-a-z-score-what-is-a-p-value.htm
- https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-statistics/h-how-hot-spot-analysis-getis-ord-gi-spatial-stati.htm

In [1]:
from itertools import product

import geopandas as gp
import pyspark.sql.functions as F
import xyzservices.providers as xyz
from pyspark.sql import SparkSession

import geofunctions as S

In [2]:
spark = (SparkSession
         .builder
         .config("spark.jars", "../target/geofunctions-0.5.jar")
         .getOrCreate())

23/08/14 09:14:19 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
pts = (
    spark.read.format("shp")
    .options(path="../data/RegressionTutorial/RegressionExercise/Data/911Calls.shp")
    .load()
)

### Get count per cell.

In [4]:
cell = 800

qr_ln = (
    pts.select(S.st_xy("shape").alias("xy"))
    .select(
        S.st_xtoq("xy.x", cell),
        S.st_ytor("xy.y", cell),
    )
    .groupBy("q", "r")
    .count()
    .withColumnRenamed("count", "c")
)

### Define a neighborhood of a cell.

In [5]:
data = list(product(range(-3, 4), range(-3, 4)))
prod = spark.createDataFrame(data, "i int,j int")

In [6]:
prod.count()

49

### Broadcast the cell to its neighbors.

In [7]:
qr_nn = (
    qr_ln
    .crossJoin(F.broadcast(prod))
    .select(F.expr("q+i as q"), F.expr("r+j as r"), "c")
    .groupBy("q", "r")
    .agg(F.sum("c").alias("n"))
)

### Get the overall statistics.

In [8]:
rows = qr_nn.select("n").summary("mean", "stddev").select("n").collect()
mu = F.lit(float(rows[0].n))
sd = F.lit(float(rows[1].n))

23/08/14 09:14:32 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
z = (
    qr_nn
    # Make sure we get only the orignal cells.
    .join(qr_ln, ["q", "r"])
    .select(
        S.st_qtox("q", cell),
        S.st_rtoy("r", cell),
        "c",
        "n",
        ((F.col("n") - mu) / sd).alias("z"),
    )
    .select(
        S.st_cell("x", "y", cell).alias("geometry"),
        F.col("c").alias("Points in Cell"),
        F.col("n").alias("Points from neighbors"),
        F.greatest(F.lit(-3.0), F.least(F.lit(3.0), "z")).alias("z"),
    )
    .orderBy("z")
)

### Visualize the Z scores.

In [10]:
pdf = z.toPandas()

In [11]:
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs="ESRI:102726")
gdf.sindex;

In [12]:
gdf.explore("z", cmap="coolwarm", vmin=-3.0, vmax=3.0, tiles=xyz.Esri.WorldGrayCanvas)